# 01 — Exploratory Data Analysis: NASA IMS Bearing Dataset

**Project:** robotic-bearing-pdm  
**Dataset:** NASA IMS Bearing Dataset 2 (2004, 4 bearings, ~984 snapshots over 7 days)  
**Goal:** Understand the raw vibration signals, identify the failure progression,  
and define the healthy training window for the LSTM Autoencoder.

---

### Dataset Background
- **Source:** NASA Intelligent Maintenance Systems (IMS) Center, University of Cincinnati  
- **Sampling rate:** 20 kHz — each file captures 1.024 seconds of vibration (20,480 samples)  
- **Snapshot interval:** ~10 minutes  
- **Dataset 2 duration:** 7 days (Feb 12–19, 2004)  
- **Known failures:** Bearing 1 → inner race defect, Bearing 2 → outer race defect  
- **Download:** https://www.kaggle.com/datasets/vinayak123tyagi/bearing-dataset  
- **Expected path:** `data/raw/2nd_test/`

## 0 · Setup

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebook
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import signal, stats

from src.data.loader import load_snapshots, train_test_split_temporal

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 4)

DATA_DIR  = ROOT / "data" / "raw"
SAVE_DIR  = ROOT / "data" / "processed"
SAVE_DIR.mkdir(exist_ok=True)

print(f"Project root : {ROOT}")
print(f"Data dir     : {DATA_DIR}")

## 1 · Load Dataset

In [ ]:
signals, timestamps, bearing_names = load_snapshots(DATA_DIR, dataset=2)

n_files, n_samples, n_bearings = signals.shape
duration_days = (timestamps[-1] - timestamps[0]).total_seconds() / 86400

print(f"Files loaded  : {n_files}")
print(f"Samples/file  : {n_samples:,}  (@ 20 kHz = {n_samples/20000:.3f} s)")
print(f"Bearings      : {n_bearings}  → {bearing_names}")
print(f"Date range    : {timestamps[0]}  →  {timestamps[-1]}")
print(f"Duration      : {duration_days:.1f} days")
print(f"Memory        : {signals.nbytes / 1e6:.1f} MB")

## 2 · Raw Signal Visualization

Compare the raw vibration waveform at three points in time:  
**early** (healthy), **mid-run**, and **near-failure**.

In [ ]:
SAMPLE_RATE = 20_000
t_axis = np.arange(n_samples) / SAMPLE_RATE  # time in seconds

snapshot_indices = {
    "Early (healthy)": 10,
    "Mid-run": n_files // 2,
    "Near failure": n_files - 20,
}

fig, axes = plt.subplots(len(snapshot_indices), n_bearings,
                          figsize=(16, 8), sharey=False)

for row, (label, idx) in enumerate(snapshot_indices.items()):
    for col, bname in enumerate(bearing_names):
        ax = axes[row, col]
        ax.plot(t_axis[:2000], signals[idx, :2000, col],
                lw=0.6, color=f"C{col}")
        ax.set_title(f"{bname}\n{label}  [{timestamps[idx].strftime('%m-%d %H:%M')}]",
                     fontsize=8)
        ax.set_xlabel("Time (s)", fontsize=7)
        ax.set_ylabel("Amplitude (g)", fontsize=7)
        ax.tick_params(labelsize=7)

plt.suptitle("Raw Vibration Waveforms — first 0.1 s of each snapshot",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(SAVE_DIR / "01_raw_waveforms.png", bbox_inches="tight")
plt.show()

## 3 · Statistical Feature Trends Over Time

For each 10-minute snapshot we compute scalar health indicators.  
These will become the feature matrix fed into the LSTM Autoencoder.

| Feature | What it captures |
|---|---|
| **RMS** | Overall energy — rises as bearing deteriorates |
| **Kurtosis** | Impulsiveness — spikes sharply when surface defects produce impacts |
| **Crest Factor** | Peak/RMS ratio — sensitive to early-stage pitting |
| **Peak-to-Peak** | Signal range — increases with looseness and spalling |
| **Skewness** | Asymmetry — changes when bearing geometry is damaged |

In [ ]:
def extract_time_features(sig: np.ndarray) -> dict:
    """Compute scalar health features for a single 1-D signal array."""
    rms = np.sqrt(np.mean(sig ** 2))
    peak = np.max(np.abs(sig))
    return {
        "rms"          : rms,
        "kurtosis"     : float(stats.kurtosis(sig)),
        "crest_factor" : peak / (rms + 1e-9),
        "peak_to_peak" : float(np.ptp(sig)),
        "skewness"     : float(stats.skew(sig)),
        "std"          : float(np.std(sig)),
    }

# Build feature DataFrame — one row per snapshot, one column per (bearing × feature)
records = []
for i, ts in enumerate(timestamps):
    row = {"timestamp": ts}
    for col, bname in enumerate(bearing_names):
        feats = extract_time_features(signals[i, :, col])
        row.update({f"{bname}_{k}": v for k, v in feats.items()})
    records.append(row)

df_features = pd.DataFrame(records).set_index("timestamp")
print(f"Feature matrix shape: {df_features.shape}")
df_features.head(3)

In [ ]:
# Plot RMS and Kurtosis trends for all 4 bearings
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for col, bname in enumerate(bearing_names):
    axes[0].plot(df_features.index, df_features[f"{bname}_rms"],
                 label=bname, lw=1.2)
    axes[1].plot(df_features.index, df_features[f"{bname}_kurtosis"],
                 label=bname, lw=1.2)

axes[0].set_ylabel("RMS (g)")
axes[0].set_title("RMS over Time — energy rises near failure")
axes[0].legend(loc="upper left")

axes[1].set_ylabel("Kurtosis")
axes[1].set_title("Kurtosis over Time — spikes indicate impulsive impacts (bearing defects)")
axes[1].legend(loc="upper left")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(SAVE_DIR / "02_rms_kurtosis_trends.png", bbox_inches="tight")
plt.show()

In [ ]:
# Full feature panel for Bearing 1 (the one that fails)
b1_feats = [c for c in df_features.columns if c.startswith("bearing_1")]
fig, axes = plt.subplots(len(b1_feats), 1, figsize=(14, 14), sharex=True)

for ax, feat in zip(axes, b1_feats):
    ax.plot(df_features.index, df_features[feat], lw=1.0, color="C0")
    ax.set_ylabel(feat.replace("bearing_1_", ""), fontsize=9)
    ax.tick_params(axis="x", labelsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
plt.xticks(rotation=30)
plt.suptitle("Bearing 1 — All Feature Trends (inner race failure)",
             fontsize=11, y=1.005)
plt.tight_layout()
plt.savefig(SAVE_DIR / "03_bearing1_all_features.png", bbox_inches="tight")
plt.show()

## 4 · FFT Spectrum Analysis

The frequency spectrum reveals **bearing defect frequencies** — characteristic peaks
appear at Ball Pass Frequency Outer race (BPFO) and Ball Pass Frequency Inner race (BPFI)
when surface defects are present.

We compare the average spectrum in the **healthy window** vs. **failure window**.

In [ ]:
HEALTHY_END  = int(n_files * 0.20)   # first 20% = training / healthy reference
FAILURE_START = n_files - 50          # last 50 snapshots = near-failure zone

def avg_fft(sig_block: np.ndarray, fs: int = 20_000) -> tuple:
    """Average magnitude FFT over a block of snapshots (n, samples)."""
    freqs, _, Sxx = signal.spectrogram(sig_block.T, fs=fs, nperseg=1024)
    return freqs, Sxx.mean(axis=-1)   # mean power over time

fig, axes = plt.subplots(1, n_bearings, figsize=(16, 4), sharey=False)

for col, bname in enumerate(bearing_names):
    healthy_block  = signals[:HEALTHY_END, :, col]
    failure_block  = signals[FAILURE_START:, :, col]

    f_h, psd_h = avg_fft(healthy_block)
    f_f, psd_f = avg_fft(failure_block)

    axes[col].semilogy(f_h, psd_h.mean(axis=0), label="Healthy",  lw=1.2, alpha=0.8)
    axes[col].semilogy(f_f, psd_f.mean(axis=0), label="Near-fail", lw=1.2, alpha=0.8)
    axes[col].set_title(bname, fontsize=9)
    axes[col].set_xlabel("Frequency (Hz)", fontsize=8)
    axes[col].set_ylabel("PSD", fontsize=8)
    axes[col].legend(fontsize=7)
    axes[col].set_xlim(0, 5000)

plt.suptitle("Average Power Spectral Density: Healthy vs Near-Failure",
             fontsize=11)
plt.tight_layout()
plt.savefig(SAVE_DIR / "04_psd_healthy_vs_failure.png", bbox_inches="tight")
plt.show()

## 5 · Failure Zone Identification

We use rolling RMS to find where each bearing starts degrading.  
The **anomaly threshold** for training = μ + 3σ computed on the healthy window.

In [ ]:
fig, axes = plt.subplots(1, n_bearings, figsize=(16, 4), sharey=False)

for col, bname in enumerate(bearing_names):
    rms_series = df_features[f"{bname}_rms"]
    rolling    = rms_series.rolling(window=20, center=True).mean()

    # Healthy window stats
    healthy_rms = rms_series.iloc[:HEALTHY_END]
    mu, sigma   = healthy_rms.mean(), healthy_rms.std()
    threshold   = mu + 3 * sigma

    ax = axes[col]
    ax.plot(rms_series.index, rms_series.values,
            alpha=0.3, color=f"C{col}", lw=0.8, label="RMS")
    ax.plot(rolling.index, rolling.values,
            color=f"C{col}", lw=1.5, label="Rolling mean (20)")
    ax.axhline(threshold, color="red", lw=1.2, ls="--",
               label=f"μ+3σ = {threshold:.4f}")
    ax.axvspan(rms_series.index[0], rms_series.index[HEALTHY_END],
               alpha=0.08, color="green", label="Training window")

    ax.set_title(bname, fontsize=9)
    ax.set_xlabel("Date", fontsize=8)
    ax.set_ylabel("RMS (g)", fontsize=8)
    ax.legend(fontsize=6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.suptitle("RMS Trend with μ+3σ Anomaly Threshold (red dashed) and Training Window (green)",
             fontsize=10)
plt.tight_layout()
plt.savefig(SAVE_DIR / "05_failure_zones.png", bbox_inches="tight")
plt.show()

print(f"\nHealthy training window : snapshots 0 → {HEALTHY_END}  "
      f"({timestamps[0].strftime('%m-%d %H:%M')} → {timestamps[HEALTHY_END].strftime('%m-%d %H:%M')})")
print(f"Test window             : snapshots {HEALTHY_END} → {n_files}")

## 6 · Correlation Heatmap

Check inter-feature and inter-bearing correlations.  
Highly correlated features won't add information to the LSTM — useful for feature selection.

In [ ]:
corr = df_features.corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=12)
plt.tight_layout()
plt.savefig(SAVE_DIR / "06_feature_correlation.png", bbox_inches="tight")
plt.show()

## 7 · Save Feature Matrix

Export the feature matrix to `data/processed/features_dataset2.csv`  
for use in the feature engineering notebook and model training.

In [ ]:
out_path = SAVE_DIR / "features_dataset2.csv"
df_features.to_csv(out_path)
print(f"Saved: {out_path}")
print(f"Shape: {df_features.shape}")
df_features.describe().round(4)

## 8 · Key Findings

| Finding | Detail |
|---|---|
| **Dataset duration** | ~7 days, ~984 snapshots @ 10 min intervals |
| **Healthy window** | First 20% of snapshots (≈ 1.4 days) — used for LSTM training |
| **Bearing 1** | Inner race failure — kurtosis and crest factor rise sharply |
| **Bearing 2** | Outer race failure — RMS increase is the strongest signal |
| **Bearings 3 & 4** | Relatively stable throughout — good negative reference |
| **Best features** | RMS and kurtosis show the clearest degradation signal |
| **Feature matrix** | Saved to `data/processed/features_dataset2.csv` |

### Next Steps
- `02_feature_engineering.ipynb` — add FFT band energy, crest factor, rolling statistics  
- `03_model_training.ipynb` — train LSTM Autoencoder on healthy window, evaluate on test set